#### Define notebook Parameters

Pass through th pl_worker

In [ ]:
# Framework-compatible parameters with manual fallbacks.
# When pl_worker calls this notebook, it should pass these values.
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load RMDD to silver"
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False

# Manual watermark values
#prev_wm = "2026-06-01 00:00:00"
prev_wm = "1900-01-01 00:00:00"
curr_wm = "2026-06-30 00:00:00"

# Manual connection fallback
server_name = "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"
source_connection_settings = "{}"

# Source config
# source_tables: first table is the main source; extra tables are used for joins
source_settings = json.dumps({
    "source_name": "CAN_RMDD",
    "database_name": "CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e",
    "table_config": [
        {
            "schema_name": "dbo",
            "source_tables": ["Address"],
            "target_table": "silver_rmdd_address",
            "primary_keys": ["address_id"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["Client"],
            "target_table": "silver_rmdd_client",
            "primary_keys": ["client_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["Person"],
            "target_table": "silver_rmdd_person",
            "primary_keys": ["person_number", "member_firm_code", "person_key"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["PersonRelationship", "Person", "RelationshipType"],
            "target_table": "silver_rmdd_person_relationship",
            "primary_keys": ["person_number_1", "member_firm_code_1", "person_number_2", "member_firm_code_2"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["PersonRelationship", "Person", "RelationshipType"],
            "target_table": "silver_rmdd_relationship_type",
            "primary_keys": ["person_number_1", "member_firm_code_1", "person_number_2", "member_firm_code_2", "relationship_description"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["Person", "PersonSystemMap"],
            "target_table": "silver_rmdd_person_system_map",
            "primary_keys": ["person_number", "member_firm_code", "personsystemmap_key"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["PersonLocale","person" ],
            "target_table": "silver_rmdd_person_locate",
            "primary_keys": ["person_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["email","person" ],
            "target_table": "silver_rmdd_email",
            "primary_keys": ["person_number", "member_firm_code","email_key"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["costcenter"],
            "target_table": "silver_rmdd_cost_centre",
            "primary_keys": ["cost_centre_code", "member_firm_code","valid_from_date"],
            "is_incremental": 1,
            "incremental_column": "TIMESTAMP"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["matter"],
            "target_table": "silver_rmdd_matter",
            "primary_keys": ["matter_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["phone","person"],
            "target_table": "silver_rmdd_phone",
            "primary_keys": ["person_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        }
    ]
})

# Target config
target_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "schema_name": "silver_rmdd",
    "load_strategy": "merge-scd1",
    "location_root": "Files/silver/rmdd"
})

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 186, Finished, Available, Finished, False)

#### Define functions and logging

Used for import functions

In [ ]:
%run nb_transformation_shared_functions

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 191, Finished, Available, Finished, True)

In [ ]:
# Set up standard framework logging
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 192, Finished, Available, Finished, False)

#### Define variables or parameters

Can manually run or be through worker

In [ ]:
# Parse notebook settings
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_name = source_settings.get("source_name")
source_database = source_settings.get("database_name")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse_name = target_settings.get("lakehouse_name")
target_schema = target_settings.get("schema_name")
target_load_strategy = target_settings.get("load_strategy", "merge")
location_root = target_settings.get("location_root")

# SQL connection configuration
server_name = (
    source_connection_settings.get("server_name")
    or source_connection_settings.get("jdbc_hostname")
    or source_connection_settings.get("server")
    or server_name
)

# Validation output
print(source_name)
print(source_database)
print(target_schema)
display(table_config)

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 193, Finished, Available, Finished, False)

CAN_RMDD
CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e
silver_rmdd


SynapseWidget(Synapse.DataFrame, 35c77d7b-06db-40ff-8acb-cd27a3dc3f40)

#### Build JDBC connection

Connects to source SQL database

In [ ]:
# Build JDBC connection to source SQL database
jdbc_url = (
    "jdbc:sqlserver://"
    f"{server_name}:1433;"
    f"database={source_database};"
    "encrypt=true;"
)

connection_properties = {
    "accessToken": mssparkutils.credentials.getToken("https://database.windows.net/"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Print JDBC URL only
print(jdbc_url)

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 194, Finished, Available, Finished, False)

jdbc:sqlserver://hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com:1433;database=CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e;encrypt=true;


#### Ingest source table

Read and dedup clean source

In [ ]:
# Find every unique source table needed by the target mappings
source_tables_to_read = {}

for cfg in table_config:
    source_schema = cfg.get("schema_name")
    source_tables = cfg.get("source_tables", [])

    if not source_tables:
        raise ValueError(f"Missing source_tables for {cfg.get('target_table')}")

    for source_table in source_tables:
        source_view = f"src_{to_snake_case(source_table)}"

        source_tables_to_read[source_view] = {
            "schema_name": source_schema,
            "source_table": source_table
        }

# Read source tables and create src_* temp views
for source_view, source_cfg in source_tables_to_read.items():
    source_schema = source_cfg.get("schema_name")
    source_table = source_cfg.get("source_table")
    full_source_name = f"{source_schema}.{source_table}"

    source_query = f"""
    (
        SELECT *
        FROM {full_source_name}
    ) AS src
    """

    # Read source table from SQL
    df_source = spark.read.jdbc(
        url=jdbc_url,
        table=source_query,
        properties=connection_properties
    )

    # Drop exact duplicates only
    df_source = remove_exact_duplicates(df_source)

    # Create source temp view
    df_source.createOrReplaceTempView(source_view)

    # Preview source data
    print(f"Source preview for {source_view} ({full_source_name})")
    display(df_source.limit(3))

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 195, Finished, Available, Finished, False)

Source preview for src_address (dbo.Address)


SynapseWidget(Synapse.DataFrame, 16767b71-a8f7-4ca6-be86-5ff4e2aa8728)

Source preview for src_client (dbo.Client)


SynapseWidget(Synapse.DataFrame, 2b612015-e126-484c-ab3d-fd99a546a34b)

...

SynapseWidget(Synapse.DataFrame, fe9c7fe7-9145-4419-8630-e3d591432a97)

Source preview for src_matter (dbo.matter)


SynapseWidget(Synapse.DataFrame, ba79f138-91ea-40f7-9ceb-3332ff4c4b73)

Source preview for src_phone (dbo.phone)


SynapseWidget(Synapse.DataFrame, 71f1a842-832b-49ad-9630-d5021b793af7)

#### Transform source table

Applies mapping and transformation as needed

In [ ]:
# Map source views into target shape
for cfg in table_config:
    # Get source tables 
    source_tables = cfg.get("source_tables", [])
    main_source_table = source_tables[0]
    main_source_view = f"src_{to_snake_case(main_source_table)}"

    # Get target tables
    target_table = cfg.get("target_table")
    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    # Build map views
    map_view = f"map_{target_entity}"

    # Build watermark filter for this target table
    watermark_col = cfg.get("incremental_column")
    is_incremental = cfg.get("is_incremental", 0)

    # Map Address
    if target_entity == "address":

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"s.{watermark_col} > '{prev_wm}' AND s.{watermark_col} <= '{curr_wm}'"
        
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY s.AddressID) AS BIGINT) AS address_key,
            s.AddressID                     AS address_id,
            s.AddressTypeCode               AS address_type_code,
            s.Address1                      AS address_1,
            s.Address2                      AS address_2,
            s.Address3                      AS address_3,
            s.Address4                      AS address_4,
            s.Address5                      AS address_5,
            s.City                          AS city,
            s.CountryID                     AS country_id,
            CAST(NULL AS STRING)            AS country_code_iso2,
            s.State                         AS state,
            s.PostalCode                    AS zip_code,
            s.Address1                      AS street_address_check,
            s.Active                        AS is_active,
            s.LastUpdatedDateGMT            AS last_updated_date_utc_at_source
        FROM {main_source_view} s
        WHERE {watermark_filter}
        """)

    # Map Client
    elif target_entity == "client":

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"s.{watermark_col} > '{prev_wm}' AND s.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY s.ClientID) AS BIGINT) AS client_key,
            s.ClientNumber                  AS client_number,
            s.MemberFirmCode                AS member_firm_code,
            s.ClientID                      AS client_id,
            s.PMSClientID                   AS pms_client_id,
            s.ClientName                    AS client_name,
            s.AlternateClientName           AS alternate_client_name,
            s.ContactName                   AS contact_name,
            s.OpenDateGMT                   AS open_date_utc,
            s.CloseDateGMT                  AS close_date_utc,
            s.ClientDUNS                    AS client_duns,
            s.ClientGUDUNS                  AS client_guduns,
            s.Comments                      AS comments,
            s.PMSEntityType                 AS pms_entity_type,
            s.ClientType                    AS client_type,
            s.IsConfidential                AS is_confidential,
            s.SICCode                       AS sic_code,
            CAST(NULL AS STRING)            AS client_sector_key,
            s.SanctionsChecked_YN           AS is_sanctions_checked,
            s.Active                        AS is_active,
            s.CreatedDateGMT                AS created_date_utc_at_source,
            s.LastUpdatedDateGMT            AS last_updated_date_utc_at_source
        FROM {main_source_view} s
        WHERE {watermark_filter}
        """)

    # Map Person
    elif target_entity == "person":

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"s.{watermark_col} > '{prev_wm}' AND s.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY s.PersonID) AS BIGINT) AS person_key,
            s.SystemEmployeeCode            AS person_number,
            s.MemberFirmCode                AS member_firm_code,
            s.PersonID                      AS person_id,
            s.PersonStatusCode              AS person_status_code,
            s.Gender                        AS gender,
            s.Prefix                        AS prefix,
            s.FirstName                     AS first_name,
            s.FirstNameKnownAs              AS first_name_known_as,
            s.MiddleName                    AS middle_name,
            s.LastName                      AS last_name,
            s.Suffix                        AS suffix,
            s.Initials                     AS initials,
            s.PhotoPath                     AS photo_path,
            s.CommencementDateGMT           AS commencement_date_utc,
            s.DepartureDateGMT              AS departure_date_utc,
            s.LeaveStatus                   AS leave_status,
            CAST(NULL AS STRING)            AS team_key,
            s.JobTitle                      AS job_title,
            s.JobTitlePMS                   AS job_title_pms,
            s.PracticeGroup                 AS practice_group_code,
            CAST(NULL AS STRING)            AS practice_group_key,
            CAST(s.YearOfCall AS STRING)    AS year_of_call,
            s.LastName                      AS employee_name_reporting,
            CAST(s.DeleteFlag AS BOOLEAN)   AS is_active,
            s.LastUpdatedDateGMT            AS last_updated_date_utc_at_source
        FROM {main_source_view} s
        WHERE {watermark_filter}
        """)

    # Map PersonRelationship
    elif target_entity == "person_relationship":
        person_relationship_view = f"src_{to_snake_case(source_tables[0])}"
        person_view = f"src_{to_snake_case(source_tables[1])}"
        relationship_type_view = f"src_{to_snake_case(source_tables[2])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"pr.{watermark_col} > '{prev_wm}' AND pr.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY pr.PersonID1, pr.PersonID2, pr.RelationshipTypeID) AS BIGINT) AS person_relationship_key,
            p1.SystemEmployeeCode           AS person_number_1,
            p1.MemberFirmCode               AS member_firm_code_1,
            p2.SystemEmployeeCode           AS person_number_2,
            p2.MemberFirmCode               AS member_firm_code_2,
            rt.RelationshipTypeName         AS relationship_description,
            pr.PrimaryFlag                  AS primary_flag,
            pr.Active                       AS is_active,
            pr.LastUpdatedDateGMT           AS last_updated_date_utc_at_source
        FROM {person_relationship_view} pr
        LEFT JOIN {person_view} p1
            ON pr.PersonID1 = p1.PersonID
        LEFT JOIN {person_view} p2
            ON pr.PersonID2 = p2.PersonID
        LEFT JOIN {relationship_type_view} rt
            ON pr.RelationshipTypeID = rt.RelationshipTypeID
        WHERE {watermark_filter}
        """)

    # Map RelationshipType
    elif target_entity == "relationship_type":
        person_relationship_view = f"src_{to_snake_case(source_tables[0])}"
        person_view = f"src_{to_snake_case(source_tables[1])}"
        relationship_type_view = f"src_{to_snake_case(source_tables[2])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"pr.{watermark_col} > '{prev_wm}' AND pr.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY pr.PersonID1, pr.PersonID2, pr.RelationshipTypeID) AS BIGINT) AS person_relationship_key,
            p1.SystemEmployeeCode           AS person_number_1,
            p1.MemberFirmCode               AS member_firm_code_1,
            p2.SystemEmployeeCode           AS person_number_2,
            p2.MemberFirmCode               AS member_firm_code_2,
            rt.RelationshipTypeName         AS relationship_description,
            pr.PrimaryFlag                  AS primary_flag,
            pr.Active                       AS is_active,
            pr.LastUpdatedDateGMT           AS last_updated_date_utc_at_source
        FROM {person_relationship_view} pr
        LEFT JOIN {person_view} p1
            ON pr.PersonID1 = p1.PersonID
        LEFT JOIN {person_view} p2
            ON pr.PersonID2 = p2.PersonID
        LEFT JOIN {relationship_type_view} rt
            ON pr.RelationshipTypeID = rt.RelationshipTypeID
        WHERE {watermark_filter}
        """)

    # Map PersonSystemMap
    elif target_entity == "person_system_map":
        person_view = f"src_{to_snake_case(source_tables[0])}"
        person_system_map_view = f"src_{to_snake_case(source_tables[1])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"p.{watermark_col} > '{prev_wm}' AND p.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY p.PersonID) AS BIGINT) AS personsystemmap_key,
            LTRIM(RTRIM(p.SystemEmployeeCode))          AS person_number,
            LTRIM(RTRIM(p.MemberFirmCode))              AS member_firm_code,
            LTRIM(RTRIM(psm.GPMSExternalPersonID))      AS timekeeper_id,
            LTRIM(RTRIM(psm.DomainName))                AS domain_name,
            LTRIM(RTRIM(psm.GPMSExternalPersonID))      AS unified_sap_person_external_id,
            LTRIM(RTRIM(psm.WindowsLogin))              AS windows_login,
            LTRIM(RTRIM(psm.DMSSystemID))               AS dms_system_id,
            LTRIM(RTRIM(psm.HumanResourcesSystemID))    AS human_resources_system_id
        FROM {person_view} p
        LEFT JOIN {person_system_map_view} psm
            ON p.PersonID = psm.PersonID
        WHERE {watermark_filter}
        """)


        # Map personlocate
    elif target_entity == "person_locate":
        personlocale_view = f"src_{to_snake_case(source_tables[0])}"
        person_view = f"src_{to_snake_case(source_tables[1])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"p.{watermark_col} > '{prev_wm}' AND p.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY p.PersonID) AS BIGINT) AS personlocale_key,
            LTRIM(RTRIM(p.SystemEmployeeCode))          AS person_number,
            LTRIM(RTRIM(psm.MemberFirmCode))              AS member_firm_code,
            LTRIM(RTRIM(psm.PersonID))                    AS person_id,
            LTRIM(RTRIM(psm.DeskLocation))              AS desk_location,
            CAST(NULL AS STRING)                        AS office_key
        FROM {person_view} p
        LEFT JOIN {personlocale_view} psm
            ON p.PersonID = psm.PersonID
        WHERE {watermark_filter}
        """)

    # Map email
    elif target_entity == "email":
        email_view = f"src_{to_snake_case(source_tables[0])}"
        person_view = f"src_{to_snake_case(source_tables[1])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"p.{watermark_col} > '{prev_wm}' AND p.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY p.PersonID) AS BIGINT) AS email_key,
            LTRIM(RTRIM(p.SystemEmployeeCode))          AS person_number,
            LTRIM(RTRIM(p.MemberFirmCode))              AS member_firm_code,
            LTRIM(RTRIM(psm.EntityID))                    AS entity_id,
            LTRIM(RTRIM(psm.EmailAddress))              AS email_address
        FROM {person_view} p
        LEFT JOIN {email_view} psm
            ON p.PersonID = psm.EntityID
        WHERE {watermark_filter}
        """)

    # Map cost_centre
    elif target_entity == "cost_centre":
        costcenter_view = f"src_{to_snake_case(source_tables[0])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"c.{watermark_col} > '{prev_wm}' AND c.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY c.CostCenter) AS BIGINT) AS cost_centre_key,
            LTRIM(RTRIM(c.CostCenter))          AS cost_centre_code,
            "GPMS"                              AS member_firm_code,
            LTRIM(RTRIM(c.ValidFromDate))       AS valid_from_date,
            LTRIM(RTRIM(c.CompanyCode))              AS company_code,
            LTRIM(RTRIM(c.CurrencyKey))              AS currency_code,
            LTRIM(RTRIM(c.ProfitCenter))              AS profit_centre,
            LTRIM(RTRIM(c.Description))              AS cost_centre_description,
            LTRIM(RTRIM(c.LongDescription))              AS cost_centre_long_description,
            LTRIM(RTRIM(c.ValidToDate))              AS valid_to_date,
            CASE WHEN LTRIM(RTRIM(c.ValidToDate))  = '9999-12-31' THEN 1 
            ELSE 0 END                              AS is_active,
            CAST(NULL AS STRING)                    AS company_key,
            CAST(NULL AS STRING)                    AS currency_key,
            c.TIMESTAMP                             AS last_updated_date_utc_at_source
        FROM {costcenter_view} c
        --WHERE {watermark_filter}
        """)

    # Map matter
    elif target_entity == "matter":
        matter_view = f"src_{to_snake_case(source_tables[0])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"m.{watermark_col} > '{prev_wm}' AND m.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY m.MatterID) AS BIGINT) AS matter_key,
            LTRIM(RTRIM(m.MatterNumber))            AS matter_number,
            LTRIM(RTRIM(m.MemberFirmCode))          AS member_firm_code,
            LTRIM(RTRIM(m.MatterID))                AS matter_id,
            LTRIM(RTRIM(m.MatterName))              AS matter_name,
            LTRIM(RTRIM(m.LongMatterName))          AS long_matter_name,
            LTRIM(RTRIM(m.AlternateMatterName))     AS alternate_matter_name,
            LTRIM(RTRIM(m.AlternateMatterNumber))   AS alternate_matter_number,
            LTRIM(RTRIM(m.NatureOfProceeding))      AS nature_of_proceeding_code,
            LTRIM(RTRIM(m.MatterStatusID))              AS matter_status_code,
            LTRIM(RTRIM(m.ClientID))              AS client_number,
            LTRIM(RTRIM(m.Confidential))              AS is_confidential,
            LTRIM(RTRIM(m.CurrencyCode))              AS currency_code,
            LTRIM(RTRIM(m.MatterOpenDateGMT))              AS matter_open_date_utc,
            LTRIM(RTRIM(m.MatterCloseDateGMT))              AS matter_close_date_utc,
            LTRIM(RTRIM(m.MatterBillableFlag))              AS matter_billable_flag,
            LTRIM(RTRIM(m.MatterTimeUnit))              AS matter_time_unit,
            LTRIM(RTRIM(m.SubIndustry))              AS matter_sector_key,
            LTRIM(RTRIM(m.TeamID))              AS team_id,
            LTRIM(RTRIM(m.Comments))              AS comments,
            LTRIM(RTRIM(m.GPMSGroupBilling))              AS group_billing,
            LTRIM(RTRIM(m.GPMSReportingGroup))              AS reporting_group,
            LTRIM(RTRIM(m.SanctionsChecked_YN))              AS is_sanctions_checked,
            LTRIM(RTRIM(m.TechnicalArea))              AS technical_area_code,
            CASE 
                WHEN m.Active = 1 THEN true
                WHEN m.Active = 0 THEN false 
                ELSE NULL
            END AS is_active,
            CAST(NULL AS STRING)                AS office_id_key,
            CAST(NULL AS STRING)                AS nature_of_proceeding_key,
            CAST(NULL AS STRING)                AS matter_status_key,
            CAST(NULL AS STRING)                AS currency_code_key,
            CAST(NULL AS STRING)                AS team_key,
            CAST(NULL AS STRING)                AS work_type_key,
            CAST(NULL AS STRING)                AS billing_office_key,
            LTRIM(RTRIM(m.PMSMatterID))         AS pms_matter_id,
            LTRIM(RTRIM(m.OfficeID))            AS office_id,
            m.LastUpdatedDateGMT                AS last_updated_date_utc_at_source

        FROM {matter_view} m
        WHERE {watermark_filter}
        """)


        # Map phone
    elif target_entity == "phone":
        phone_view = f"src_{to_snake_case(source_tables[0])}"
        person_view= f"src_{to_snake_case(source_tables[1])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"p.{watermark_col} > '{prev_wm}' AND p.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY p.PersonID) AS BIGINT) AS phone_key,
            LTRIM(RTRIM(p.SystemEmployeeCode))          AS person_number,
            LTRIM(RTRIM(p.MemberFirmCode))              AS member_firm_code,
            LTRIM(RTRIM(e.EntityID))                    AS entity_id,
            LTRIM(RTRIM(e.PhoneNumber))              AS work_phone_number,
            LTRIM(RTRIM(e.PhoneExtension))              AS work_phone_extension,
            LTRIM(RTRIM(e.PhoneNumber))              AS mobile_phone_number,
            LTRIM(RTRIM(e.PhoneTypeCode))              AS phone_type_code
        FROM {person_view} p
        LEFT JOIN {phone_view} e
            ON p.PersonID = e.EntityID
        WHERE {watermark_filter}
        """)

        


    else:
        raise ValueError(f"No mapping defined for {target_table}")

    # Preview mapped data
    print(f"Mapped preview for {map_view}")
    display(spark.table(map_view).limit(3))

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 196, Finished, Available, Finished, False)

Mapped preview for map_address


SynapseWidget(Synapse.DataFrame, e38ad2cf-da1c-403a-9acd-395c3db9aeae)

Mapped preview for map_client


SynapseWidget(Synapse.DataFrame, 26fb7d3e-deec-4f6a-8ed3-1536406db6f9)

...

SynapseWidget(Synapse.DataFrame, 715681df-62ec-4aad-ab5f-99fb28541fd7)

Mapped preview for map_matter


SynapseWidget(Synapse.DataFrame, 6ba39dad-27c8-4d3d-8591-0cc920a0d644)

Mapped preview for map_phone


SynapseWidget(Synapse.DataFrame, 6ab6f835-76c5-4f7a-82a6-0d9ca0f18f73)

#### Add metadata to source table

Applies metadata and _hashed_pk

In [ ]:
# Add metadata and _hashed_pk to mapped views
# This creates final tgt_* views used by validation and merge
for cfg in table_config:
    source_schema = cfg.get("schema_name")
    source_tables = cfg.get("source_tables", [])
    target_table = cfg.get("target_table")
    primary_keys = cfg.get("primary_keys", [])

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    map_view = f"map_{target_entity}"
    target_view = f"tgt_{target_entity}"
    source_file = ", ".join([f"{source_schema}.{source_table}" for source_table in source_tables])

    # Read mapped temp view
    df_target = spark.table(map_view)

    # Build hashed primary key
    pk_expr_cols = [to_snake_case(col) for col in primary_keys]

    # Create _hashed_pk from primary key
    df_target = df_target.withColumn(
        "_hashed_pk",
        F.md5(F.concat_ws("|", *[F.col(col).cast("string") for col in pk_expr_cols]))
    )

    # Get non-business columns to build hashed row
    row_hash_cols = [
        col for col in df_target.columns
        if col not in [df_target.columns[0], "_hashed_pk"] + pk_expr_cols
    ]

    # Create _hashed_row from non-business columns
    df_target = df_target.withColumn(
        "_hashed_row",
        F.md5(F.concat_ws("|", *[F.coalesce(F.col(col).cast("string"), F.lit("<NULL>")) for col in row_hash_cols]))
    )

    # Add framework metadata
    df_target = add_metadata(
        df=df_target,
        ingest_date=str(date.today()),
        file_path=source_file,
        schema_name=source_name,
        table_name=target_table,
        lineage_id=lineage_id
    )

    # Create final target temp view
    df_target.createOrReplaceTempView(target_view)

    # Preview final data
    print(f"Final preview for {target_view}")
    display(df_target.limit(3))

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 197, Finished, Available, Finished, False)

Final preview for tgt_address


SynapseWidget(Synapse.DataFrame, e746efca-9ec6-4d17-98b7-cc3c36e3323c)

Final preview for tgt_client


SynapseWidget(Synapse.DataFrame, 86ca3f60-dfb1-4223-acba-e3466393e56a)

...

SynapseWidget(Synapse.DataFrame, eb48cf5f-2148-4915-9149-8fceeffe6e09)

Final preview for tgt_matter


SynapseWidget(Synapse.DataFrame, f8a1edd9-deae-486d-a303-3d4b21b535cd)

Final preview for tgt_phone


SynapseWidget(Synapse.DataFrame, fc6cef4b-2e37-4f39-a983-4b4af87e407f)

#### Check duplicates

Return duplicates before merge 

In [ ]:
# Check duplicate hashed keys before merge
for cfg in table_config:
    target_table = cfg.get("target_table")

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    target_view = f"tgt_{target_entity}"

    # Validate duplicate primary keys
    print(f"Checking duplicates for {target_view}")
    validate_duplicates(spark.table(target_view), "_hashed_pk", 10)

    # to do later
    # DO NOT TRIGGER ERROR IF DUPLICATS, CONTINUE BUT SAVE RECONDS ON dupliate tABLE


StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 198, Finished, Available, Finished, False)

Checking duplicates for tgt_address


Total duplicate ['_hashed_pk'] groups: 0
Checking duplicates for tgt_client


Total duplicate ['_hashed_pk'] groups: 0
Checking duplicates for tgt_person


Total duplicate ['_hashed_pk'] groups: 0
Checking duplicates for tgt_person_relationship
Total duplicate ['_hashed_pk'] groups: 0
Checking duplicates for tgt_relationship_type
Total duplicate ['_hashed_pk'] groups: 0
Checking duplicates for tgt_person_system_map
Total duplicate ['_hashed_pk'] groups: 0
Checking duplicates for tgt_person_locate


...

SynapseWidget(Synapse.DataFrame, 7d0d12e3-c8a5-4f46-9ce9-9c5ce64a8272)

Checking duplicates for tgt_matter


Total duplicate ['_hashed_pk'] groups: 0
Checking duplicates for tgt_phone


Total duplicate ['_hashed_pk'] groups: 2790


SynapseWidget(Synapse.DataFrame, 11c77f9c-5acd-445c-8492-60a704ca5e4e)

#### Merge to target

Merge to target table based on _hashed_key

In [ ]:
# Merge final temp views into existing target silver tables
load_results = []

for cfg in table_config:
    target_table = cfg.get("target_table")

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    target_view = f"tgt_{target_entity}"
    target_table_path = f"{location_root}/{target_table}"
    full_target_name = f"{target_schema}.{target_table}"

    # Run target load
    df_target = spark.table(target_view)
    result = load_to_target(df_target, full_target_name, target_load_strategy, target_table_path)
    
    # Save metrics for pipeline
    load_results.append({
        "table": full_target_name,
        "rows_inserted": result["rows_inserted"],
        "rows_updated": result["rows_updated"],
        "rows_read": result["rows_read"],
        "rows_copied": result["rows_copied"]
    })

display(load_results)

StatementMeta(, 4c65d55d-3480-43b1-ac97-e757a424b6c1, 199, Finished, Available, Finished, True)

2026-07-02 21:20:55,729 UTC - INFO - Running merge-scd1 into silver_rmdd.silver_rmdd_address (LoadTransactionalToBase)


2026-07-02 21:21:03,027 UTC - INFO - Skipping silver_rmdd.silver_rmdd_address: no changes (LoadTransactionalToBase)
2026-07-02 21:21:03,047 UTC - INFO - Running merge-scd1 into silver_rmdd.silver_rmdd_client (LoadTransactionalToBase)


2026-07-02 21:21:12,795 UTC - INFO - Skipping silver_rmdd.silver_rmdd_client: no changes (LoadTransactionalToBase)
2026-07-02 21:21:12,813 UTC - INFO - Running merge-scd1 into silver_rmdd.silver_rmdd_person (LoadTransactionalToBase)


2026-07-02 21:21:18,786 UTC - INFO - Skipping silver_rmdd.silver_rmdd_person: no changes (LoadTransactionalToBase)
2026-07-02 21:21:18,804 UTC - INFO - Running merge-scd1 into silver_rmdd.silver_rmdd_person_relationship (LoadTransactionalToBase)


...

2026-07-02 21:21:24,866 UTC - INFO - Skipping silver_rmdd.silver_rmdd_person_system_map: no changes (LoadTransactionalToBase)
2026-07-02 21:21:24,913 UTC - INFO - Running merge-scd1 into silver_rmdd.silver_rmdd_person_locate (LoadTransactionalToBase)


2026-07-02 21:21:27,977 UTC - INFO - Skipping silver_rmdd.silver_rmdd_person_locate: no changes (LoadTransactionalToBase)
2026-07-02 21:21:27,995 UTC - INFO - Running merge-scd1 into silver_rmdd.silver_rmdd_email (LoadTransactionalToBase)


2026-07-02 21:21:31,115 UTC - INFO - Skipping silver_rmdd.silver_rmdd_email: no changes (LoadTransactionalToBase)
2026-07-02 21:21:31,143 UTC - INFO - Running merge-scd1 into silver_rmdd.silver_rmdd_cost_centre (LoadTransactionalToBase)


2026-07-02 21:21:38,931 UTC - INFO - Running merge-scd1 into silver_rmdd.silver_rmdd_matter (LoadTransactionalToBase)


Py4JJavaError: An error occurred while calling o761404.execute.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.sql.delta.perf.DeltaOptimizedWriterExec.awaitShuffleMapStage$1(DeltaOptimizedWriterExec.scala:157)
	at org.apache.spark.sql.delta.perf.DeltaOptimizedWriterExec.getShuffleStats(DeltaOptimizedWriterExec.scala:162)
	at org.apache.spark.sql.delta.perf.DeltaOptimizedWriterExec.computeBins(DeltaOptimizedWriterExec.scala:104)
	at org.apache.spark.sql.delta.perf.DeltaOptimizedWriterExec.doExecute(DeltaOptimizedWriterExec.scala:178)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$execute$1(SparkPlan.scala:220)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$executeQuery$1(SparkPlan.scala:271)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at